# 课后练习解答（06.06_comparison_and_conclusion）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** 优化后吞吐提升 15%、测量噪声 ±5%，正确结论是？
A. 有提升趋势，但需更多 repeats/置信度验证
B. 必然提升
C. 完全无效
D. 只能说明显存降低

**解答：** A

**解析：** 提升幅度超过噪声但仍需统计验证，不能仅凭单次实验下结论。


### 问题2（单选题）

**题目：** 优化后时延下降但峰值显存不变，最可能原因？
A. 减少了 kernel 启动与中间读写，但中间张量峰值未下降
B. 参数减少
C. 精度下降
D. 数据增强

**解答：** A

**解析：** 融合减少调度和读写，但同一时刻仍可能有完整中间特征图，峰值显存未必下降。


### 问题3（多选题）

**题目：** 对比报告必须说明？
A. 硬件型号与软件版本
B. batch_size 与输入
C. warmup/repeats 与同步
D. 结论适用范围

**解答：** ABCD

**解析：** 缺少环境与测量方法，性能结论不可复现、不可比较。


### 问题4（多选题）

**题目：** 融合后还可叠加的优化？
A. FP16/BF16
B. CANN 图优化
C. 更大 batch
D. 增加 BN 层

**解答：** ABC

**解析：** 增加 BN 与融合目标相反，会重新引入额外算子。


### 问题5（判断题）

**题目：** baseline 与 optimized 使用不同 batch_size 也可以直接比较吞吐。

**解答：** 错

**解析：** 不同 batch 下启动开销占比与访存模式不同，结果不可比。


### 问题6（判断题）

**题目：** 模型层融合与 CANN/ATC 图优化可以叠加。

**解答：** 对

**解析：** 模型层融合先消除 BN，ATC 编译时仍可继续做图级融合与调度优化。


### 问题7（填空题）

**题目：** 相对提升率公式：speedup = ____。

**解答：** (baseline_ms - optimized_ms) / baseline_ms * 100%


### 问题8（填空题）

**题目：** 报告应记录设备型号、____、____ 等环境信息。

**解答：** CANN/torch_npu 版本；Python 版本


### 问题9（简答题）

**题目：** 融合后没有明显收益的可能原因有哪些？

**解答：** batch 过小使 kernel 启动占比高但设备本身已做图融合、测量噪声大、模型已 memory-bound 而融合收益有限，或对比时未固定 warmup/repeats 等变量。


### 问题10（简答题）

**题目：** 如何量化融合前后数值误差？

**解答：** 固定同一输入，在 eval/no_grad 下分别前向，计算输出的最大绝对误差、最大相对误差和 shape 是否一致；FP32 下可接受 1e-4~1e-2 量级，同时报告误差分布而非只看最大值。


### 问题11（代码设计题）

**题目：** 编写 compare_results(baseline.json, optimized.json)，输出各指标相对变化百分比。

**解答：** ```python
def compare_results(baseline_path, optimized_path):
    base = json.load(open(baseline_path))
    opt = json.load(open(optimized_path))
    out = {}
    for key in ["ms_per_iter", "throughput", "peak_memory_mb"]:
        if key in base and key in opt:
            out[key + "_pct"] = (opt[key] - base[key]) / base[key] * 100
    return out
```


### 问题12（单选题）

**题目：** 优化后时延下降且精度未变，可以认为？
A. 该优化在本实验设定下有效
B. 优化无效
C. 必须重训
D. 数据错误

**解答：** A

**解析：** 性能提升且正确性保持，即满足本次实验的优化目标。


### 问题13（多选题）

**题目：** 对比实验中需要保持的变量包括？
A. 输入分布
B. batch_size
C. warmup/repeats
D. 设备状态

**解答：** ABCD

**解析：** 变量控制越严格，性能差异越能归因于优化本身。


### 问题14（判断题）

**题目：** 单次 batch 的测量结果足以作为最终性能结论。

**解答：** 错

**解析：** 单次测量受缓存、频率和调度波动影响，必须多次测量取统计结果。


### 问题15（简答题）

**题目：** 总结一套可复用于其他算子的优化方法论。

**解答：** 定位热点算子 → 分析算术强度与瓶颈类型 → 寻找数学等价变换或调度优化 → 建立可复现 baseline → 实施融合/低精度/Tiling → 正确性校验 → 多次测量对比 → 结合 CANN 图优化与精度要求评估收益。
